In [1]:
from notebook import * 

/home/bhan001/cs211-hw2-solving-large-linear-system-private-bugparty/notebook.py:427: SyntaxWarning: invalid escape sequence '\s'
  show = (f"[\s\*]{re.escape(show)}\s*\(", "^\}")
/home/bhan001/cs211-hw2-solving-large-linear-system-private-bugparty/notebook.py:427: SyntaxWarning: invalid escape sequence '\s'
  show = (f"[\s\*]{re.escape(show)}\s*\(", "^\}")
/home/bhan001/cs211-hw2-solving-large-linear-system-private-bugparty/notebook.py:427: SyntaxWarning: invalid escape sequence '\}'
  show = (f"[\s\*]{re.escape(show)}\s*\(", "^\}")
/home/bhan001/cs211-hw2-solving-large-linear-system-private-bugparty/notebook.py:429: SyntaxWarning: invalid escape sequence '\s'
  show = (f"^{re.escape(show)}:\s*", ".cfi_endproc")


Done loading notebook! We're good to go!


# Q1

$$ 
A = \begin{pmatrix}
1 & 2 & 3 \\
4 & 13 & 18 \\
7 & 54 & 78
\end{pmatrix}
$$


$$ r_2-4r_1   \rightarrow
 \begin{pmatrix}
1 & 2 & 3 \\
0 & 5 & 6 \\
7 & 54 & 78
\end{pmatrix}
$$

$$ r_3-7r_1   \rightarrow
 \begin{pmatrix}
1 & 2 & 3 \\
0 & 5 & 6 \\
0 & 40 & 57
\end{pmatrix}
$$

$$ r_3-8r_2   \rightarrow
A_4 = \begin{pmatrix}
1 & 2 & 3 \\
0 & 5 & 6 \\
0 & 0 & 9
\end{pmatrix}
$$

$$ U = \begin{pmatrix}
1 & 2 & 3 \\
0 & 5 & 6 \\
0 & 0 & 9
\end{pmatrix}$$

$$ I = \begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{pmatrix}$$

$$ r_3+8r_2 \rightarrow \begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 8 & 1
\end{pmatrix}$$

$$ r_3+7r_1 \rightarrow \begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
7 & 8 & 1
\end{pmatrix}$$

$$ r_2+4r_1  \rightarrow \begin{pmatrix}
1 & 0 & 0 \\
4 & 1 & 0 \\
7 & 8 & 1
\end{pmatrix}$$

$$ L  \rightarrow \begin{pmatrix}
1 & 0 & 0 \\
4 & 1 & 0 \\
7 & 8 & 1
\end{pmatrix}$$

$$ A = \begin{pmatrix}
1 & 0 & 0 \\
4 & 1 & 0 \\
7 & 8 & 1
\end{pmatrix}\begin{pmatrix}
1 & 2 & 3 \\
0 & 5 & 6 \\
0 & 0 & 9
\end{pmatrix} = L\times U$$

Q2

In [1]:
! make bench
! srun bench

rm -f bench
gcc main.c -o bench -I /act/opt/intel/composer_xe_2013.3.163/mkl/include -L /act/opt/intel/composer_xe_2013.3.163/mkl/lib/intel64 -O3 -march=native -fprefetch-loop-arrays -DMKL_ILP64 -lmkl_avx2 -lmkl_intel_lp64 -lmkl_sequential -lmkl_core -lpthread -lm -m64
bench: error while loading shared libraries: libmkl_avx2.so: cannot open shared object file: No such file or directory
srun: error: cluster-001-compute-001: task 0: Exited with exit code 127


the core of the computation is here.
```
for (i=0;i<n-1;++i){
for(j=i+1; j<n; ++j){
            A[j*n+i] /=  A[i*n+i];
            for(k=i+1;k<n;++k){
                A[j*n+k] -= A[j*n+i]*A[i*n+k];
            }
        }
}
```
total flops will be
$$\sum_{i=0}^{n-2}(\sum_{j=i+1}^{n-1}(1+\sum_{k=i+1}^{n-1}\times 2))$$
simplified, we got
$$ \frac{n(n-1)(4n+1)}{6}$$

Q4

naive gemm 6s
matrix size 2048
block size 64
gemm2_ikj with kernel gemm2_kernel_ikj 1.733647s
gemm2_ikj with kernel gemm2_kernel_ijk  time=5.221353s
gemm2_kij with kernel gemm2_kernel_ikj time=1.798215s
gemm2_kij with kernel gemm2_kernel_ijk time=5.256797s


block size 32
gemm2_ikj with kernel gemm2_kernel_ikj time=1.474892s
gemm2_ikj with kernel gemm2_kernel_ijk  time=3.529859s
gemm2_kij with kernel gemm2_kernel_ikj time=1.466103s
gemm2_kij with kernel gemm2_kernel_ijk time=3.308022s

block size 16

gemm2_ikj with kernel gemm2_kernel_ikj time=1.850689s
gemm2_ikj with kernel gemm2_kernel_ijk time=2.014001s

gemm2_kij with kernel gemm2_kernel_ikj time=1.856311s
gemm2_kij with kernel gemm2_kernel_ijk time=2.023650s


block size 8

gemm2_ikj with kernel gemm2_kernel_ikj time=2.161841s

gemm2_ikj with kernel gemm2_kernel_ijk time=1.038547s

gemm2_kij with kernel gemm2_kernel_ikj time=2.164994s
gemm2_kij with kernel gemm2_kernel_ijk time=1.043894s

my best blocked vs unblocked
n=1024, pad=1
time=0.142120s
n=1024, pad=1
time=0.133448s
n=2048, pad=1
time=1.419077s
n=2048, pad=1
time=1.045721s
n=3072, pad=1
time=6.314619s
n=3072, pad=1
time=3.471887s
n=4096, pad=1
time=15.780349s
n=4096, pad=1
time=10.072522s
n=5120, pad=1
time=31.659635s
n=5120, pad=1
time=16.001343s
